# LoRA fine-tune for self-improving-coder (free Kaggle / Colab GPU)

1. Kaggle: *New Notebook* → Settings → Accelerator **GPU T4 x2** (or P100), Internet **on**. Colab: Runtime → T4 GPU.
2. Run all cells (~10–20 min for a few hundred examples).
3. Download `qwen-sic-lora.gguf` (or the merged folder) and import it locally with Ollama (last cell shows how).

The training data is `data/finetune/experience_sft.jsonl`: verified solutions and repair traces from the **train split only**.

In [ ]:
!git clone -q https://github.com/gambrelcolby63/self-improving-coder.git
%cd self-improving-coder
!pip install -q "transformers>=4.45" "peft>=0.13" "trl>=0.12" datasets accelerate
!wc -l data/finetune/experience_sft.jsonl

In [ ]:
!python scripts/finetune_lora.py --data data/finetune/experience_sft.jsonl --out lora-out --epochs 2 --merge

## Export to GGUF for Ollama (optional, on the GPU box)

In [ ]:
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python llama.cpp/convert_hf_to_gguf.py lora-out-merged --outfile qwen-sic-f16.gguf --outtype f16
# quantize to Q4_K_M needs the llama-quantize binary (cmake build), or keep f16 (~3 GB)

## Back on your machine
```
# Modelfile: FROM ./qwen-sic-f16.gguf  (copy TEMPLATE/PARAMETER lines from the repo's Modelfile)
ollama create sic-qwen-ft -f Modelfile.ft
SIC_MODEL=sic-qwen-ft python -m evals.run_eval --cond b   # write to a fresh results dir first
```
Promote it only if it beats the current champion in `results/champion.json`.